In [1]:
import pandas as pd
import numpy as np
import re
import os


In [2]:
BRONZE_FILE = os.path.join("bronze", "airbnb_raw_data.csv")
SILVER_DIR = "silver"
SILVER_FULL_FILE = os.path.join(SILVER_DIR, "airbnb_clean_data.csv")
SILVER_STAR_FILE = os.path.join(SILVER_DIR, "airbnb_star_ready.csv")
os.makedirs(SILVER_DIR, exist_ok=True)

df = pd.read_csv(BRONZE_FILE)
print(df.shape)
df.head()


(353, 16)


,listing_name,city,country,price_raw,property_type,guests,bedrooms,beds,baths,host_name,is_superhost,is_guest_favorite,rating,reviews,scrape_date,url
0,Lovely Apartment with Parking,Amsterdam,Netherlands,NaN,Entire rental unit,8.0,3.0,3.0,3.0,EricNew HostEnjoy your stay in,1,0,NaN,NaN,2026-07-07,https://www.airbnb.com/rooms/17227175490672516...
1,Light apartment next to Vondelpark,Amsterdam,Netherlands,NaN,Entire rental unit,2.0,2.0,1.0,1.0,Kimo En Marieke,1,0,4.8,5.0,2026-07-07,https://www.airbnb.com/rooms/14711794?search_m...
2,Modern Apartment Parking,Amsterdam,Netherlands,NaN,Entire rental unit,7.0,3.0,3.0,3.0,StefanNew HostListing highligh,1,0,NaN,NaN,2026-07-07,https://www.airbnb.com/rooms/17218898431659363...
3,Stay tuned,Amsterdam,Netherlands,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,2026-07-07,https://www.airbnb.com/rooms/70363368112394565...
4,2 Floor Apartment with Large kitchen Nieuw-Vennep,Amsterdam,Netherlands,NaN,Entire rental unit,5.0,2.0,3.0,1.0,Chris,1,1,4.96,96139.0,2026-07-07,https://www.airbnb.com/rooms/92675576315069414...


In [3]:
df = df.drop_duplicates(subset=["url"]).reset_index(drop=True)
print(df.shape)


(353, 16)


In [4]:
PROPERTY_CATEGORY_MAP = {
    "Entire rental unit": "Apartment",
    "Entire condo": "Apartment",
    "Entire serviced apartment": "Apartment",
    "Entire loft": "Apartment",
    "Entire guest suite": "Apartment",
    "Entire home": "House",
    "Entire home/apt": "House",
    "Entire vacation home": "House",
    "Entire villa": "House",
    "Entire chalet": "House",
}

df["property_category"] = df["property_type"].map(PROPERTY_CATEGORY_MAP).fillna("Unknown")
df["property_category"].value_counts()


property_category
Apartment    321
House         23
Unknown        9
Name: count, dtype: int64

In [5]:
HOST_BADGE_MARKERS = [
    "Superhost",
    "New Host",
    "Guest favorite",
    "Listing highlight",
    "Enjoy your stay",
    "Welcome to",
]

HOST_SPLIT_PATTERN = re.compile("|".join(re.escape(m) for m in HOST_BADGE_MARKERS))


def clean_host_name(name):
    if pd.isna(name):
        return None
    match = HOST_SPLIT_PATTERN.search(name)
    cleaned = name[: match.start()] if match else name
    cleaned = cleaned.strip()
    return cleaned if cleaned else None


df["host_name"] = df["host_name"].apply(clean_host_name)
df["host_name"] = df["host_name"].fillna("Unknown Host")
df["host_name"].tail(15)


338                  Val
339                  Val
340               Lorenz
341             Wilfried
342        Vienna Jungle
343              Manfred
344                 HOME
345               Nevena
346                  Moe
347               Polina
348                  Max
349               Dmytro
350    EST Residence Sch
351                Mario
352                 Anna
Name: host_name, dtype: object

In [6]:
df["is_new_listing"] = (df["rating"] == "New").astype(int)
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df["reviews"] = df["reviews"].fillna(0).astype(int)
df[["rating", "is_new_listing", "reviews"]].head(10)


,rating,is_new_listing,reviews
0,NaN,0,0
1,4.80,0,5
2,NaN,0,0
3,NaN,0,0
4,4.96,0,96139
5,4.77,0,770
6,5.00,0,2
7,4.99,0,9997
8,NaN,0,0
9,NaN,0,0


In [7]:
CURRENCY_SYMBOL_MAP = {
    "$": "USD",
    "\u00a3": "GBP",
    "\u20ac": "EUR",
}

CURRENCY_CODE_PATTERN = re.compile(r"\b(NOK|EUR|USD|GBP|CHF|CZK|HUF|SEK|DKK|PLN)\b")
NUMBER_PATTERN = re.compile(r"[\d][\d.,\s]*\d|\d")


def parse_amount(number_str):
    number_str = number_str.strip()
    has_comma = "," in number_str
    has_dot = "." in number_str
    has_space = " " in number_str

    if has_comma and has_dot:
        number_str = number_str.replace(",", "")
    elif has_comma:
        decimals = number_str.split(",")[-1]
        if len(decimals) == 2:
            number_str = number_str.replace(",", ".")
        else:
            number_str = number_str.replace(",", "")
    elif has_dot:
        decimals = number_str.split(".")[-1]
        if len(decimals) == 3:
            number_str = number_str.replace(".", "")

    if has_space:
        number_str = number_str.replace(" ", "")

    try:
        return float(number_str)
    except ValueError:
        return None


def parse_price(raw):
    if pd.isna(raw):
        return None, None

    text = raw.strip()

    currency = None
    for symbol, code in CURRENCY_SYMBOL_MAP.items():
        if symbol in text:
            currency = code
            break

    if currency is None:
        code_match = CURRENCY_CODE_PATTERN.search(text)
        if code_match:
            currency = code_match.group(1)

    number_match = NUMBER_PATTERN.search(text)
    if not number_match:
        return None, currency

    amount = parse_amount(number_match.group(0))
    return amount, currency


parsed = df["price_raw"].apply(parse_price)
df["price_amount"] = parsed.apply(lambda x: x[0])
df["price_currency"] = parsed.apply(lambda x: x[1]).fillna("EUR")
df[["price_raw", "price_amount", "price_currency"]].dropna(subset=["price_amount"]).head(10)


,price_raw,price_amount,price_currency
5,€ 300,300.0,EUR
13,€25.00,25.0,EUR
20,€40,40.0,EUR
21,€15,15.0,EUR
23,€ 150,150.0,EUR
24,€ 50,50.0,EUR
29,€75,75.0,EUR
33,€5.90,5.9,EUR
35,€50,50.0,EUR
38,€350,350.0,EUR


In [8]:
EUR_CONVERSION_RATE = {
    "EUR": 1.0,
    "USD": 0.92,
    "GBP": 1.17,
    "NOK": 0.085,
    "CHF": 1.04,
    "CZK": 0.040,
    "HUF": 0.0025,
    "SEK": 0.087,
    "DKK": 0.134,
    "PLN": 0.23,
}

df["price_eur"] = df.apply(
    lambda row: row["price_amount"] * EUR_CONVERSION_RATE.get(row["price_currency"], 1.0)
    if pd.notna(row["price_amount"])
    else None,
    axis=1,
)
df[["price_raw", "price_amount", "price_currency", "price_eur"]].dropna(subset=["price_eur"]).head(10)


,price_raw,price_amount,price_currency,price_eur
5,€ 300,300.0,EUR,300.0
13,€25.00,25.0,EUR,25.0
20,€40,40.0,EUR,40.0
21,€15,15.0,EUR,15.0
23,€ 150,150.0,EUR,150.0
24,€ 50,50.0,EUR,50.0
29,€75,75.0,EUR,75.0
33,€5.90,5.9,EUR,5.9
35,€50,50.0,EUR,50.0
38,€350,350.0,EUR,350.0


In [9]:
MIN_PLAUSIBLE_PRICE_EUR = 20

df["price_flagged_invalid"] = df["price_eur"] < MIN_PLAUSIBLE_PRICE_EUR
df.loc[df["price_flagged_invalid"], "price_eur"] = None

print(df["price_flagged_invalid"].sum(), "prices flagged as implausible and nulled")
print(df["price_eur"].notna().sum(), "valid prices remaining out of", len(df))


29 prices flagged as implausible and nulled
63 valid prices remaining out of 353


In [10]:
city_median_price = df.groupby("city")["price_eur"].median()
global_median_price = df["price_eur"].median()

MIN_SAMPLES_PER_CITY = 3
city_valid_counts = df.groupby("city")["price_eur"].apply(lambda s: s.notna().sum())

df["is_price_imputed"] = df["price_eur"].isna()

def impute_price(row):
    if pd.notna(row["price_eur"]):
        return row["price_eur"]
    if city_valid_counts.get(row["city"], 0) >= MIN_SAMPLES_PER_CITY:
        return city_median_price.get(row["city"], global_median_price)
    return global_median_price

df["price_eur"] = df.apply(impute_price, axis=1)
df["price_eur"] = df["price_eur"].round(2)
df[["city", "price_eur", "is_price_imputed"]].head(10)


,city,price_eur,is_price_imputed
0,Amsterdam,62.5,True
1,Amsterdam,62.5,True
2,Amsterdam,62.5,True
3,Amsterdam,62.5,True
4,Amsterdam,62.5,True
5,Amsterdam,300.0,False
6,Amsterdam,62.5,True
7,Amsterdam,62.5,True
8,Amsterdam,62.5,True
9,Amsterdam,62.5,True


In [11]:
df["bedrooms"] = df["bedrooms"].fillna(2)

guests_by_bedrooms = df.groupby("bedrooms")["guests"].median()
beds_by_bedrooms = df.groupby("bedrooms")["beds"].median()
baths_by_bedrooms = df.groupby("bedrooms")["baths"].median()

df["guests"] = df.apply(
    lambda r: guests_by_bedrooms.get(r["bedrooms"], df["guests"].median()) if pd.isna(r["guests"]) else r["guests"],
    axis=1,
)
df["beds"] = df.apply(
    lambda r: beds_by_bedrooms.get(r["bedrooms"], df["beds"].median()) if pd.isna(r["beds"]) else r["beds"],
    axis=1,
)
df["baths"] = df.apply(
    lambda r: baths_by_bedrooms.get(r["bedrooms"], df["baths"].median()) if pd.isna(r["baths"]) else r["baths"],
    axis=1,
)

df["bedrooms"] = df["bedrooms"].astype(int)
df["beds"] = df["beds"].round().astype(int)
df["guests"] = df["guests"].round().astype(int)
df["baths"] = df["baths"].round(1)

df[["bedrooms", "guests", "beds", "baths"]].isna().sum()


bedrooms    0
guests      0
beds        0
baths       0
dtype: int64

In [12]:
silver_df = df[[
    "listing_name",
    "city",
    "country",
    "price_eur",
    "is_price_imputed",
    "price_flagged_invalid",
    "property_type",
    "property_category",
    "guests",
    "bedrooms",
    "beds",
    "baths",
    "host_name",
    "is_superhost",
    "is_guest_favorite",
    "rating",
    "is_new_listing",
    "reviews",
    "scrape_date",
    "url",
]]

print(silver_df.shape)
silver_df.isna().sum()


(353, 20)


listing_name              0
city                      0
country                   0
price_eur                 0
is_price_imputed          0
price_flagged_invalid     0
property_type             9
property_category         0
guests                    0
bedrooms                  0
beds                      0
baths                     0
host_name                 0
is_superhost              0
is_guest_favorite         0
rating                   80
is_new_listing            0
reviews                   0
scrape_date               0
url                       0
dtype: int64

In [13]:
silver_df.to_csv(SILVER_FULL_FILE, index=False)
print(SILVER_FULL_FILE)
silver_df.head()


silver\airbnb_clean_data.csv


,listing_name,city,country,price_eur,is_price_imputed,price_flagged_invalid,property_type,property_category,guests,bedrooms,beds,baths,host_name,is_superhost,is_guest_favorite,rating,is_new_listing,reviews,scrape_date,url
0,Lovely Apartment with Parking,Amsterdam,Netherlands,62.5,True,False,Entire rental unit,Apartment,8,3,3,3.0,Eric,1,0,NaN,0,0,2026-07-07,https://www.airbnb.com/rooms/17227175490672516...
1,Light apartment next to Vondelpark,Amsterdam,Netherlands,62.5,True,False,Entire rental unit,Apartment,2,2,1,1.0,Kimo En Marieke,1,0,4.80,0,5,2026-07-07,https://www.airbnb.com/rooms/14711794?search_m...
2,Modern Apartment Parking,Amsterdam,Netherlands,62.5,True,False,Entire rental unit,Apartment,7,3,3,3.0,Stefan,1,0,NaN,0,0,2026-07-07,https://www.airbnb.com/rooms/17218898431659363...
3,Stay tuned,Amsterdam,Netherlands,62.5,True,False,NaN,Unknown,4,2,3,1.5,Unknown Host,0,0,NaN,0,0,2026-07-07,https://www.airbnb.com/rooms/70363368112394565...
4,2 Floor Apartment with Large kitchen Nieuw-Vennep,Amsterdam,Netherlands,62.5,True,False,Entire rental unit,Apartment,5,2,3,1.0,Chris,1,1,4.96,0,96139,2026-07-07,https://www.airbnb.com/rooms/92675576315069414...


In [14]:
star_ready_df = silver_df.drop(columns=["property_type", "price_flagged_invalid"])

print(star_ready_df.shape)
star_ready_df.isna().sum()


(353, 18)


listing_name          0
city                  0
country               0
price_eur             0
is_price_imputed      0
property_category     0
guests                0
bedrooms              0
beds                  0
baths                 0
host_name             0
is_superhost          0
is_guest_favorite     0
rating               80
is_new_listing        0
reviews               0
scrape_date           0
url                   0
dtype: int64

In [15]:
star_ready_df.to_csv(SILVER_STAR_FILE, index=False)
print(SILVER_STAR_FILE)
star_ready_df.head()


silver\airbnb_star_ready.csv


,listing_name,city,country,price_eur,is_price_imputed,property_category,guests,bedrooms,beds,baths,host_name,is_superhost,is_guest_favorite,rating,is_new_listing,reviews,scrape_date,url
0,Lovely Apartment with Parking,Amsterdam,Netherlands,62.5,True,Apartment,8,3,3,3.0,Eric,1,0,NaN,0,0,2026-07-07,https://www.airbnb.com/rooms/17227175490672516...
1,Light apartment next to Vondelpark,Amsterdam,Netherlands,62.5,True,Apartment,2,2,1,1.0,Kimo En Marieke,1,0,4.80,0,5,2026-07-07,https://www.airbnb.com/rooms/14711794?search_m...
2,Modern Apartment Parking,Amsterdam,Netherlands,62.5,True,Apartment,7,3,3,3.0,Stefan,1,0,NaN,0,0,2026-07-07,https://www.airbnb.com/rooms/17218898431659363...
3,Stay tuned,Amsterdam,Netherlands,62.5,True,Unknown,4,2,3,1.5,Unknown Host,0,0,NaN,0,0,2026-07-07,https://www.airbnb.com/rooms/70363368112394565...
4,2 Floor Apartment with Large kitchen Nieuw-Vennep,Amsterdam,Netherlands,62.5,True,Apartment,5,2,3,1.0,Chris,1,1,4.96,0,96139,2026-07-07,https://www.airbnb.com/rooms/92675576315069414...
